### Importance of text pre processing
* What is text preprocessing
    * Invovles cleaning and perparing raw text data to make it suitable for machine learning models
    * Critical Steps for chieving high performance in NLP tasks
    * Key steps
        * Tokenization: Splits text into individual units eg words, sentences
            * eg: I Love NLP: [i , love, NLP]
        * Stemming : reduces words to their root form by removing suffixes
            * Eg: "running", "runner", -> "run"
        * Lemmeatization: Converts words to their base form using vocabulary
            * Eg: better - > good

* What are Word Embeddings
    * Dense vector representation of words that capture semantic meaning
    * Represent Words in a continuous vector space
    * Popular Word Embedding Models
        * Word2Vec:
            * Model: Continuous Bag of Words (CBOW) and skip-gram
            * Capture word relationships based on context
        * GloVe(Global Vectors for word representation)
            * USes word co-occurence statistics to generate embeddings
            * Represntational global semantic relationships
        * Pre-trained Embeddings in Framewors
            * Frameworks like tensorflow and pytorch offer pre-trained embeddings for quick intergration
    * Benefits of Word Embedding
        * reduces dimensionality
        * captures semantic similarity
        * imprioves model generalizations

* W#h y use pretrained embeddings
    * saves computational resources
    * Leverages large, well-trained models
    * boosts performance for downstream task
* popularte pre-trained embeddings
    * GloVe: Pre-Trained on datasets like wikipedia and common crawl
    * Fast Text : Handles out of vocabulary (OOV) words through subword embeddings
    * Embedding Layers in deep learning frameworks like pytorch and tensorflow         


In [9]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
!wget https://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2026-06-15 16:08:01--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-06-15 16:08:01--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  4.97MB/s    in 2m 39s  

2026-06-15 16:10:40 (5.18 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]

Archive:  glove.6B.zip
  inflating: glove.6B.50d.txt        
  inflating: glove.6B.100d.txt       
  inflatin

In [11]:
!cp glove.6B.100d.txt /content/drive/MyDrive/

In [13]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM

In [3]:
VOCAB_SIZE = 10000
MAX_LEN = 200

(X_train,y_train,), (X_test,y_test)= imdb.load_data(num_words=MAX_LEN)

# Decode reviews to text for preprocessing
word_index = imdb.get_word_index()
reverse_word_index = {value:key for key, value in word_index.items()}
decoded_reviews = [" ".join([reverse_word_index.get(i-2,"?") for i in review]) for review in X_train[:5]]

# padd_sequences
X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding="post")
X_test = pad_sequences(X_test,maxlen=MAX_LEN, padding="post")

print(f"Training data shape: {X_train.shape}, {y_train.shape}")
print(f"Test Data shape: {X_test.shape}, {y_test.shape}")

Training data shape: (25000, 200), (25000,)
Test Data shape: (25000, 200), (25000,)


In [7]:
print(X_train)

[[  5  25 100 ...  19 178  32]
 [  1 194   2 ...   0   0   0]
 [  1  14  47 ...   0   0   0]
 ...
 [  1  11   6 ...   0   0   0]
 [  1   2   2 ...   0   0   0]
 [  1  17   6 ...   0   0   0]]


In [15]:
# Load GloVe embeddings
embeddings_index = {}
glove_file = "glove.6B.100d.txt"
with open(glove_file, "r", encoding="utf-8") as file:
    for line in file:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1: ], dtype= "float32")
        embeddings_index[word]=coefs
print(f"Loaded {len(embeddings_index)} word vectors")

#  Prepare embedding matrix
embedding_dim = 100
embedding_matrix = np.zeros((VOCAB_SIZE, embedding_dim))

for word, i in word_index.items():
    if i < VOCAB_SIZE:
        embedding_vector = embeddings_index .get(word)
        if embedding_vector is not None:
            embedding_matrix[i]= embedding_vector

Loaded 400000 word vectors


In [16]:
#  Define the LSTM Model with GloVe embeddings
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=embedding_dim, weights=[embedding_matrix],trainable=False),
    LSTM(128, activation="tanh", return_sequences=False),
    Dense(1, activation="sigmoid"),
])

# compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# Train the model

history = model.fit(
    X_train,y_train,validation_split=0.2,epochs=10, batch_size=64, verbose = 1)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,000,000 (3.81 MB)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 133s 417ms/step - accuracy: 0.5096 - loss: 0.6942 - val_accuracy: 0.5062 - val_loss: 0.6935
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 140s 445ms/step - accuracy: 0.5234 - loss: 0.6907 - val_accuracy: 0.5336 - val_loss: 0.6883
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 134s 420ms/step - accuracy: 0.5244 - loss: 0.6894 - val_accuracy: 0.5398 - val_loss: 0.6837
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 141s 451ms/step - accuracy: 0.5461 - loss: 0.6822 - val_accuracy: 0.5112 - val_loss: 0.6962
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 138s 440ms/step - accuracy: 0.5567 - loss: 0.6760 - val_accuracy: 0.5480 - val_loss: 0.6828
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 140s 448ms/step - accuracy: 0.5701 - loss: 0.6725 - val_accuracy: 0.5532 - val_loss: 0.6730
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 130s 414ms/step - accuracy: 0.5586 - loss: 0.6752 - val_accuracy: 0.5566 - val_loss: 0.6736
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 143s 419ms/step - accuracy: 0.6140 -

In [17]:
loss, accuracy = model.evaluate(X_test,y_test, verbose=0)
print(f"LSTM Model with GloVe Test Accuracy: {accuracy}")

model.summary()

LSTM Model with GloVe Test Accuracy: 0.6905999779701233


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 100)       │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,352,133 (5.16 MB)

 Trainable params: 117,377 (458.50 KB)

 Non-trainable params: 1,000,000 (3.81 MB)

 Optimizer params: 234,756 (917.02 KB)